In [1]:
import os


!pip install llama-index-llms-groq
!pip install llama-index
!pip install llama-index-embeddings-huggingface

In [2]:
import os

# Make sure to set your GROQ_API_KEY in Colab's secrets or replace 'YOUR_GROQ_API_KEY' with your actual key
# To add to Colab secrets, click on the key icon in the left sidebar.
# If you're running this locally, you can set it as an environment variable or directly here.
os.environ["GROQ_API_KEY"] = "" # Replace with your actual Groq API key

# It's good practice to avoid hardcoding API keys directly in the notebook for security.
# For example:
# from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [3]:
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import llama_index.core

In [11]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    load_index_from_storage,
    StorageContext,
)
from IPython.display import Markdown, display
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Download Data

In [13]:
!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'

--2026-03-12 01:26:51--  https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 75042 (73K) [text/plain]
Saving to: ‘data/paul_graham/paul_graham_essay.txt’

data/paul_graham/pa 100%[===================>]  73.28K  --.-KB/s    in 0.007s  

2026-03-12 01:26:51 (10.4 MB/s) - ‘data/paul_graham/paul_graham_essay.txt’ saved [75042/75042]



In [14]:
# load documents
documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

In [37]:
# load documents
documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

# Initialize Groq LLM
llm = Groq(model="llama-3.3-70b-versatile") # Updated to a widely available Groq model

# Initialize the embedding model
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Create the VectorStoreIndex with the Groq LLM and specify the embedding model
index = VectorStoreIndex.from_documents(documents, llm=llm, embed_model=embed_model)

In [38]:
# save index to disk
index.set_index_id("vector_index")
index.storage_context.persist("./storage")

In [39]:
# rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir="./storage")
# load index
index = load_index_from_storage(storage_context, index_id="vector_index", embed_model=embed_model)

Query Index

In [40]:
# set Logging to DEBUG for more detailed outputs
# Configure the query engine to use the Groq LLM
query_engine = index.as_query_engine(llm=llm, response_mode="tree_summarize")
response = query_engine.query("What did the author do growing up?")

In [41]:
display(Markdown(f"<b>{response}</b>"))

<b>The author worked on two main things outside of school: writing and programming. The author wrote short stories, which they describe as "awful" with hardly any plot, and tried writing programs on an IBM 1401 using an early version of Fortran. They also later worked on microcomputers, including a TRS-80, where they wrote simple games, a program to predict model rocket flight, and a word processor.</b>

Query Index with SVM/Linear Regression

In [42]:
query_modes = [
    "svm",
    "linear_regression",
    "logistic_regression",
]
for query_mode in query_modes:
    # set Logging to DEBUG for more detailed outputs
    # Configure the query engine to use the Groq LLM
    query_engine = index.as_query_engine(llm=llm, vector_store_query_mode=query_mode)
    response = query_engine.query("What did the author do growing up?")
    print(f"Query mode: {query_mode}")
    display(Markdown(f"<b>{response}</b>"))

Query mode: svm


<b>The author was a painting student and attended the Rhode Island School of Design (RISD). They also learned to program and worked as a Lisp hacker. Additionally, the author took classes, including a color class and a painting class with Idelle Weber. The author did not have a natural talent for drawing, but they enjoyed painting and continued to teach themselves.</b>

Query mode: linear_regression


<b>The author was not one of the kids who could draw in high school, but they did attend the Rhode Island School of Design (RISD) to learn to paint. Prior to that, there is no information provided about the author's specific activities or hobbies growing up, except that they later developed an interest in painting.</b>

Query mode: logistic_regression


<b>The author was a painting student and attended the Rhode Island School of Design (RISD). They also took a color class and learned to paint, but did not have a natural talent for drawing like some of their peers. Additionally, the author learned Lisp programming language and worked as a freelance Lisp hacker.</b>

In [43]:
display(Markdown(f"<b>{response}</b>"))

<b>The author was a painting student and attended the Rhode Island School of Design (RISD). They also took a color class and learned to paint, but did not have a natural talent for drawing like some of their peers. Additionally, the author learned Lisp programming language and worked as a freelance Lisp hacker.</b>

In [44]:
print(response.source_nodes[0].text)

Painting students were supposed to express themselves, which to the more worldly ones meant to try to cook up some sort of distinctive signature style.

A signature style is the visual equivalent of what in show business is known as a "schtick": something that immediately identifies the work as yours and no one else's. For example, when you see a painting that looks like a certain kind of cartoon, you know it's by Roy Lichtenstein. So if you see a big painting of this type hanging in the apartment of a hedge fund manager, you know he paid millions of dollars for it. That's not always why artists have a signature style, but it's usually why buyers pay a lot for such work. [6]

There were plenty of earnest students too: kids who "could draw" in high school, and now had come to what was supposed to be the best art school in the country, to learn to draw even better. They tended to be confused and demoralized by what they found at RISD, but they kept going, because painting was what they d

Query Index with custom embedding string

In [45]:
from llama_index.core import QueryBundle

In [47]:
query_bundle = QueryBundle(
    query_str="What did the author do growing up?",
    custom_embedding_strs=["The author grew up painting."],
)
# We must pass the Groq LLM to the query engine to avoid the OpenAI default
query_engine = index.as_query_engine(llm=llm)
response = query_engine.query(query_bundle)

In [48]:
display(Markdown(f"<b>{response}</b>"))

<b>The author painted and attended the Accademia, where they had a nude model and painted still lives in their bedroom at night. They also attended RISD, but dropped out in 1993 to pursue painting on their own. Additionally, the author worked at a company called Interleaf, writing code in a Lisp dialect, before deciding to focus on painting and writing books about Lisp.</b>

Use maximum marginal relevance

In [50]:
query_engine = index.as_query_engine(
    llm=llm,
    vector_store_query_mode="mmr",
    vector_store_kwargs={"mmr_threshold": 0.2}
)
response = query_engine.query("What did the author do growing up?")

Get Sources

In [51]:
print(response.get_formatted_sources())

> Source (Doc id: d710f98d-cb3a-4cf5-9cb7-1ba250296e55): What I Worked On

February 2021

Before college the two main things I worked on, outside of schoo...

> Source (Doc id: b8540973-b3b0-48d9-967e-a84d562ba88a): Painting students were supposed to express themselves, which to the more worldly ones meant to tr...


Query Index with Filters

In [52]:
from llama_index.core import Document

doc = Document(text="target", metadata={"tag": "target"})

index.insert(doc)

In [53]:
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters

filters = MetadataFilters(
    filters=[ExactMatchFilter(key="tag", value="target")]
)

retriever = index.as_retriever(
    similarity_top_k=20,
    filters=filters,
)

source_nodes = retriever.retrieve("What did the author do growing up?")

In [54]:
# retrieves only our target node, even though we set the top k to 20
print(len(source_nodes))

1


In [55]:
print(source_nodes[0].text)
print(source_nodes[0].metadata)

target
{'tag': 'target'}
